<a href="https://colab.research.google.com/github/Innovatewithapple/Pipelines-CrossValidation-Regularisation/blob/main/regressionPipelinePractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,GridSearchCV,cross_validate,cross_val_predict
from sklearn.preprocessing import StandardScaler,OneHotEncoder,PowerTransformer
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [2]:
df = pd.read_csv('/content/advertising.csv')
df.sample(5)

,TV,Radio,Newspaper,Sales
185,205.0,45.1,19.6,22.6
57,136.2,19.2,16.6,13.2
189,18.7,12.1,23.4,6.7
109,255.4,26.9,5.5,19.8
60,53.5,2.0,21.4,8.1


In [3]:
df.skew()

,0
TV,-0.069853
Radio,0.094175
Newspaper,0.894720
Sales,-0.073739


In [6]:
x = df.drop(columns='Sales')
y = df['Sales']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [7]:
pipeline = Pipeline([
    ('scaler',StandardScaler()),
    ('model',LinearRegression())
])

In [8]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])

In [10]:
cross_score = cross_validate(pipeline,x_train,y_train,cv=5,scoring=[
    "neg_mean_absolute_error",
    "neg_mean_squared_error",
    "r2"
])
cross_score

{'fit_time': array([0.00760508, 0.0122478 , 0.00606751, 0.00641537, 0.00702691]),
 'score_time': array([0.01024914, 0.00623035, 0.00569463, 0.00599456, 0.0067091 ]),
 'test_neg_mean_absolute_error': array([-1.31790097, -1.26289836, -1.56769033, -1.13154868, -1.32854646]),
 'test_neg_mean_squared_error': array([-2.91707956, -2.63731299, -3.4632876 , -2.18329793, -3.96856387]),
 'test_r2': array([0.87620545, 0.88592805, 0.87119676, 0.92292763, 0.85099336])}

In [12]:
cross_score['test_r2'].mean()

np.float64(0.8814502491023604)

In [13]:
pipeline_ridge = Pipeline([
    ('scaler',StandardScaler()),
    ('model',Ridge())
])

In [14]:
pipeline_ridge.fit(x_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', Ridge())])

In [16]:
cross_score_ridge = cross_validate(pipeline_ridge,x_train,y_train,cv=5,scoring=[
    "neg_mean_absolute_error",
    "neg_mean_squared_error",
    "r2"
])
cross_score_ridge

{'fit_time': array([0.01203609, 0.00640011, 0.00539207, 0.00539446, 0.0056057 ]),
 'score_time': array([0.00727749, 0.00498414, 0.00489092, 0.00482941, 0.00707936]),
 'test_neg_mean_absolute_error': array([-1.32407313, -1.26048905, -1.57877928, -1.12989167, -1.34144644]),
 'test_neg_mean_squared_error': array([-2.90373991, -2.6115651 , -3.47505963, -2.15744087, -4.02255659]),
 'test_r2': array([0.87677156, 0.88704172, 0.87075894, 0.92384041, 0.84896611])}

In [17]:
cross_score_ridge['test_r2'].mean()

np.float64(0.8814757480502282)

In [18]:
pipeline_Lasso = Pipeline([
    ('scaler',StandardScaler()),
    ('model',Lasso())
])

In [19]:
pipeline_Lasso.fit(x_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', Lasso())])

In [20]:
cross_score_Lasso = cross_validate(pipeline_Lasso,x_train,y_train,cv=5,scoring=[
    "neg_mean_absolute_error",
    "neg_mean_squared_error",
    "r2"
])
cross_score_Lasso

{'fit_time': array([0.02687502, 0.00733924, 0.00677156, 0.00823903, 0.00643158]),
 'score_time': array([0.03450751, 0.00574446, 0.01288962, 0.00623631, 0.00608802]),
 'test_neg_mean_absolute_error': array([-1.58013301, -1.52498905, -2.12709648, -1.41368373, -1.91435201]),
 'test_neg_mean_squared_error': array([-3.91033958, -3.46088459, -6.51861835, -4.13445113, -6.59778536]),
 'test_r2': array([0.83405365, 0.85030603, 0.75756585, 0.85405018, 0.75227466])}

In [22]:
cross_score_Lasso['test_r2'].mean()

np.float64(0.8096500717198232)

In [23]:
grid_param = {
    "model__alpha":[0.01,0.1,1,10],
    "model__solver":['auto','svd','cholesky']
}

In [24]:
grid = GridSearchCV(pipeline_ridge,grid_param,cv=5,scoring='neg_mean_squared_error')

In [25]:
grid.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', Ridge())]),
             param_grid={'model__alpha': [0.01, 0.1, 1, 10],
                         'model__solver': ['auto', 'svd', 'cholesky']},
             scoring='neg_mean_squared_error')

In [27]:
best = grid.best_estimator_
print(best)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', Ridge(alpha=0.1, solver='svd'))])


In [28]:
y_pred = best.predict(x_test)

In [29]:
print('mse',mean_squared_error(y_test,y_pred))

mse 2.5437654050458556
